In [0]:
from pyspark.sql import functions as F

CATALOG = 'northbridge'
RAW     = f'/Volumes/{CATALOG}/raw'
STAGING = f'{RAW}/staging/by_day'     # where week 2 wrote the day folders
LANDING = f'{RAW}/landing'            # simulated "arrival" folder

SOURCES = {                           # folder name -> bronze table
    'transactions': f'{CATALOG}.bronze.transactions',
    'ledger':       f'{CATALOG}.bronze.ledger',
    'customers':    f'{CATALOG}.bronze.customers',
}

In [0]:
def land_day(source, day):
    src = f'{STAGING}/{source}/day={day}'
    dst = f'{LANDING}/{source}/day={day}'
    dbutils.fs.cp(src, dst, recurse=True)
    print('landed', source, 'day', day)

def land_all(day):
    land_day('transactions', day)
    land_day('ledger', day)
    if day in (1, 15):               # customer extracts exist for days 1 and 15 only
        land_day('customers', day)

In [0]:
def ingest(source):
    table = SOURCES[source]
    q = (spark.readStream.format('cloudFiles')
        .option('cloudFiles.format', 'csv')
        .option('header', True)
        .option('cloudFiles.schemaLocation', f'{RAW}/_schemas/{source}')
        .load(f'{LANDING}/{source}')
        .withColumn('_ingest_ts', F.current_timestamp())
        .withColumn('_source_file', F.col('_metadata.file_path'))
        .writeStream
        .option('checkpointLocation', f'{RAW}/_checkpoints/bronze_{source}')
        .trigger(availableNow=True)
        .toTable(table))
    q.awaitTermination()
    print(source, '->', table, spark.table(table).count(), 'rows')

def ingest_all():
    for s in SOURCES:
        ingest(s)

In [0]:
for d in range(8, 32):
    land_day('transactions', d)
    land_day('ledger', d)
ingest('transactions')
ingest('ledger')

In [0]:
land_day('customers', 15)
ingest('customers')

display(spark.sql("""SELECT day, count(*) AS rows FROM northbridge.bronze.customers
                     GROUP BY day ORDER BY day"""))

In [0]:
for d in range(1, 8):
    land_day('transactions', d)
    land_day('ledger', d)
ingest('transactions')
ingest('ledger')

In [0]:
land_all(1)
ingest_all()

In [0]:
land_all(2)
ingest_all()

display(spark.sql(f"""
  SELECT day, count(*) AS rows, count(DISTINCT _source_file) AS files
  FROM {SOURCES['transactions']} GROUP BY day ORDER BY day"""))

In [0]:
ingest_all()

In [0]:
for d in range(3, 8):
    land_all(d)
    ingest_all()

In [0]:
src = (spark.read.parquet('/Volumes/northbridge/raw/staging/tx_master')
       .groupBy('day').count().withColumnRenamed('count', 'source_rows'))

brz = (spark.table('northbridge.bronze.transactions')
       .withColumn('day', F.col('day').cast('int'))
       .groupBy('day').count().withColumnRenamed('count', 'bronze_rows'))

display(src.join(brz, 'day').withColumn('extra', F.col('bronze_rows') - F.col('source_rows')).orderBy('day'))

In [0]:
display(spark.sql("""
  SELECT count(*) AS total, count(DISTINCT txn_id) AS distinct_ids,
         sum(CASE WHEN txn_id IS NULL THEN 1 ELSE 0 END) AS null_ids
  FROM northbridge.bronze.transactions"""))

In [0]:
RAW  = '/Volumes/northbridge/raw'
BASE = f'{RAW}/staging'

for t in ['transactions', 'ledger']:
    spark.sql(f'DROP TABLE IF EXISTS northbridge.bronze.{t}')
    dbutils.fs.rm(f'{RAW}/landing/{t}', recurse=True)
    dbutils.fs.rm(f'{RAW}/_checkpoints/bronze_{t}', recurse=True)
    dbutils.fs.rm(f'{RAW}/_schemas/{t}', recurse=True)
    dbutils.fs.rm(f'{BASE}/by_day/{t}', recurse=True)
dbutils.fs.rm(f'{BASE}/tx_master', recurse=True)

In [0]:
display(spark.sql("""
  SELECT count(*) AS total, count(DISTINCT txn_id) AS distinct_ids
  FROM northbridge.bronze.ledger"""))